# APIM ❤️ Microsoft Web IQ

## Track Microsoft Web IQ REST and MCP usage with Azure API Management

Route [Microsoft Web IQ](https://webiq.microsoft.ai/documentation/overview/) REST and MCP requests through Azure API Management (APIM), block Browse at the gateway, and attribute request volume and latency to APIM subscriptions. Callers provide a Web IQ API key or Entra ID token; APIM does not create or inject an upstream credential. This notebook deploys the shared gateway and exercises both interfaces.

**Audience:** Developers and platform teams operating web-grounded applications.

**Prerequisites:**

- Microsoft Web IQ limited-access approval and an API key from Web IQ Profile Management.
- Python 3.12+, the repository environment installed with `uv sync`, and VS Code with the Jupyter extension.
- Azure CLI installed and authenticated.
- An Azure subscription with Contributor + RBAC Administrator, or Owner, permissions.

**Learning goals:**

- Pass a caller-provided Web IQ API key through APIM without storing it at the gateway.
- Use APIM subscriptions as consumer identities.
- Block the Browse REST operation and MCP tool before either reaches Web IQ.
- Discover and invoke Web IQ tools over streamable HTTP MCP.
- Optionally authenticate Web IQ with a Microsoft Entra ID app-only token.
- Emit and query privacy-conscious usage and latency metrics in Application Insights.

```mermaid
flowchart LR
    Client[Client application]
    MicrosoftEntra[Microsoft Entra ID]
    subgraph Gateway[Azure API Management]
        Subscription[Validate APIM subscription]
        RequestMetric[Emit request metric]
        Operation{REST operation or MCP tool?}
        BlockMetric[Emit blocked-request metric]
        Forbidden[Return structured 403]
        Auth{Web IQ credential present?}
        Unauthorized[Return structured 401]
        Entra[Pass through Entra bearer token]
        ResponseMetric[Emit response and latency metrics]
        ErrorMetric[Emit gateway-error metric]
    end
    Client -.->|Client credentials<br/>Web IQ scope| MicrosoftEntra
    MicrosoftEntra -.->|Bearer token| Client
    Client -->|APIM subscription key<br/>x-apikey or bearer token| Subscription
    Subscription --> RequestMetric --> Operation
    Operation -->|REST Browse or MCP browse| BlockMetric --> Forbidden --> Client
    Operation -->|Web Search or allowed MCP traffic| Auth
    Auth -->|x-apikey| WebIQ[Microsoft Web IQ REST and MCP]
    Auth -->|Bearer| Entra --> WebIQ
    Auth -->|Missing| Unauthorized --> Client
    WebIQ --> ResponseMetric --> Client
    Subscription -.->|Policy or gateway failure| ErrorMetric
    ErrorMetric --> Client
    RequestMetric -.-> AppInsights[Application Insights]
    BlockMetric -.-> AppInsights
    ResponseMetric -.-> AppInsights
    ErrorMetric -.-> AppInsights
    AppInsights --> Logs[Log Analytics]
```

> APIM does not store the Web IQ credential. This notebook keeps the API key only in kernel memory and sends it in the `x-apikey` request header.


## Outline

1. Configure the lab and verify Azure CLI access.
2. Deploy APIM, Application Insights, and Log Analytics.
3. Send Web IQ REST requests through APIM subscriptions.
4. Optionally call Web IQ with Microsoft Entra ID.
5. Discover and invoke Web IQ tools over MCP.
6. Verify that APIM blocks Browse over MCP.
7. Query request, response, latency, and MCP tool metrics, then exercise the REST Browse denial.


<a id='initialize'></a>
### 0️⃣ Initialize notebook variables

The caller-provided Web IQ key is read from `WEBIQ_API_KEY` when available; otherwise the notebook prompts without echoing it. APIM clients receive separate subscription keys for usage attribution.


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from getpass import getpass
from dotenv import load_dotenv

sys.path.insert(1, '../../shared')
import utils

load_dotenv()
deployment_name = 'web-iq'
resource_group_name = f'lab-{deployment_name}'
resource_group_location = 'westus2'

apim_sku = 'Basicv2'
web_iq_api_path = 'web-iq'
apim_subscriptions_config = [
    {'name': 'research-team', 'displayName': 'Research Team'},
    {'name': 'support-team', 'displayName': 'Support Team'},
]

web_iq_api_key = os.getenv('WEBIQ_API_KEY') or getpass('Microsoft Web IQ API key: ')
if not web_iq_api_key:
    raise ValueError('Set WEBIQ_API_KEY or enter a Web IQ API key when prompted.')

utils.print_ok('Notebook initialized')


<a id='azure-cli'></a>
### 1️⃣ Verify Azure CLI and the active subscription

Confirm that subsequent deployment commands target the intended tenant and subscription.


In [ ]:
output = utils.run('az account show', 'Retrieved Azure account', 'Failed to get the current Azure account')

if not output.success or not output.json_data:
    raise RuntimeError('Authenticate with Azure CLI by running az login, then rerun this cell.')

current_user = output.json_data['user']['name']
tenant_id = output.json_data['tenantId']
subscription_id = output.json_data['id']
utils.print_info(f'Current user: {current_user}')
utils.print_info(f'Tenant ID: {tenant_id}')
utils.print_info(f'Subscription ID: {subscription_id}')


<a id='deploy'></a>
### 2️⃣ Deploy the lab with Bicep

The deployment creates APIM, two APIM subscriptions, Application Insights, and Log Analytics. It does not deploy or store a Web IQ credential. The API policy permits authenticated Web Search and MCP traffic, but returns `403 Forbidden` for Browse through either interface.


In [ ]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'apimSku': {'value': apim_sku},
        'apimSubscriptionsConfig': {'value': apim_subscriptions_config},
        'webIqApiPath': {'value': web_iq_api_path},
    },
}

with open('params.json', 'w', encoding='utf-8') as parameters_file:
    json.dump(bicep_parameters, parameters_file)

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} '
    '--template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed",
)
if not output.success:
    raise RuntimeError('Deployment failed. Review the Azure CLI output above.')


<a id='outputs'></a>
### 3️⃣ Retrieve gateway details

Only the APIM subscription keys are displayed, masked to their final four characters. The caller-provided Web IQ key remains only in notebook kernel memory.


In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} --resource-group {resource_group_name}',
    f"Retrieved deployment '{deployment_name}'",
    f"Failed to retrieve deployment '{deployment_name}'",
)
if not output.success or not output.json_data:
    raise RuntimeError('Could not retrieve the deployment outputs.')

apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM gateway URL')
application_insights_name = utils.get_deployment_output(output, 'applicationInsightsName', 'Application Insights name')
web_iq_api_path = utils.get_deployment_output(output, 'webIqApiPath', 'Web IQ API path')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))

for subscription in apim_subscriptions:
    utils.print_info(f"{subscription['displayName']}: ****{subscription['key'][-4:]}")


<a id='search'></a>
### 4️⃣ Send Web Search requests through APIM

Each team uses its own APIM subscription key for usage attribution and sends its Web IQ API key in `x-apikey`. APIM strips the subscription credential, forwards the caller-provided Web IQ credential, and emits usage metrics. The compact display follows the official [Web Response schema](https://webiq.microsoft.ai/documentation/api-reference/web/#web-response), including query signals, instrumentation availability, and per-result metadata.


In [ ]:
import requests

def summarize_web_response(data: dict, content_preview_length: int = 280) -> dict:
    web_results = []
    for result in data.get('webResults', []):
        content = (result.get('content') or '').replace('\n', ' ')
        web_results.append({
            'title': result.get('title'),
            'url': result.get('url'),
            'contentPreview': content[:content_preview_length],
            'crawledAt': result.get('crawledAt'),
            'lastUpdatedAt': result.get('lastUpdatedAt'),
            'language': result.get('language'),
            'isAdult': result.get('isAdult'),
            'contentTier': result.get('contentTier'),
            'clickUrl': result.get('clickUrl'),
            'instrumentationSuffix': result.get('instrumentationSuffix'),
        })
    return {
        'traceId': data.get('traceId'),
        'querySignals': data.get('querySignals'),
        'hasInstrumentationClickBase': bool(data.get('instrumentationClickBase')),
        'hasInstrumentationCitationBase': bool(data.get('instrumentationCitationBase')),
        'webResults': web_results,
    }

search_url = f'{apim_resource_gateway_url}/{web_iq_api_path}/search/web'
workloads = [
    ('research-team', 'How does Azure API Management support AI gateways?'),
    ('support-team', 'What is Microsoft Web IQ?'),
]
subscription_by_name = {item['name']: item for item in apim_subscriptions}
request_summary = []

for subscription_name, query in workloads:
    subscription = subscription_by_name[subscription_name]
    started = time.perf_counter()
    response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'x-apikey': web_iq_api_key,
            'content-type': 'application/json',
        },
        json={
            'query': query,
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=30,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000
    utils.print_response_code(response)

    try:
        data = response.json()
    except requests.JSONDecodeError:
        data = {'rawResponse': response.text[:500]}

    web_response_summary = summarize_web_response(data) if isinstance(data, dict) else {'webResults': []}
    results = web_response_summary['webResults']
    request_summary.append({
        'subscription': subscription_name,
        'status': response.status_code,
        'latency_ms': round(elapsed_ms, 1),
        'results': len(results),
        'trace_id': web_response_summary.get('traceId'),
        'freshness': (web_response_summary.get('querySignals') or {}).get('freshness'),
        'click_instrumentation': web_response_summary.get('hasInstrumentationClickBase', False),
        'citation_instrumentation': web_response_summary.get('hasInstrumentationCitationBase', False),
    })

    print(json.dumps(web_response_summary, indent=2))
    if not response.ok:
        print(json.dumps(data, indent=2)[:2000])

request_summary


<a id='entra-id'></a>
### 5️⃣ Optional: authenticate Web IQ with Microsoft Entra ID

Web IQ recommends [Entra ID app-only authentication](https://webiq.microsoft.ai/documentation/authentication/#entra-id) for production workloads. Create an app registration and client credential, then bind its **Application (client) ID** in Web IQ Profile Management. Set `WEBIQ_TENANT_ID`, `WEBIQ_CLIENT_ID`, and `WEBIQ_CLIENT_SECRET` in your environment before running this cell. The token scope is `https://api.microsoft.ai/.default`.

The APIM subscription key still identifies the consuming team. When APIM sees the bearer token, it removes any `x-apikey` and lets Web IQ validate the Entra token. If the environment variables are absent, this optional step is skipped.


In [ ]:
from msal import ConfidentialClientApplication

entra_mcp_headers = None
entra_settings = {
    'tenant_id': os.getenv('WEBIQ_TENANT_ID'),
    'client_id': os.getenv('WEBIQ_CLIENT_ID'),
    'client_secret': os.getenv('WEBIQ_CLIENT_SECRET'),
}

if not all(entra_settings.values()):
    utils.print_info(
        'Optional Entra ID call skipped. Set WEBIQ_TENANT_ID, WEBIQ_CLIENT_ID, and WEBIQ_CLIENT_SECRET to run it.'
    )
else:
    entra_client = ConfidentialClientApplication(
        client_id=entra_settings['client_id'],
        client_credential=entra_settings['client_secret'],
        authority=f"https://login.microsoftonline.com/{entra_settings['tenant_id']}",
    )
    token_result = entra_client.acquire_token_for_client(
        scopes=['https://api.microsoft.ai/.default']
    )
    if 'access_token' not in token_result:
        raise RuntimeError(token_result.get('error_description', 'Could not acquire a Web IQ access token.'))

    entra_mcp_headers = {
        'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
        'Authorization': f"Bearer {token_result['access_token']}",
    }
    entra_response = requests.post(
        search_url,
        headers={
            'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
            'Authorization': f"Bearer {token_result['access_token']}",
            'content-type': 'application/json',
        },
        json={
            'query': 'What is Microsoft Web IQ?',
            'maxResults': 3,
            'maxLength': 3000,
            'contentFormat': 'markdown',
        },
        timeout=30,
    )
    utils.print_response_code(entra_response)
    entra_data = entra_response.json()
    print(json.dumps(summarize_web_response(entra_data), indent=2))


<a id='mcp'></a>
### 6️⃣ Connect to Web IQ over MCP

The MCP client connects to APIM's `/mcp` operation using the streamable HTTP transport, initializes a session, and discovers the tools enabled for the Web IQ credential. It uses the Entra token from the optional step when available; otherwise it sends the caller-provided `x-apikey`. The APIM subscription key continues to identify the consumer in both cases.

This section requires the repository's pinned `mcp==1.21.2` package. If the next cell reports a kernel mismatch, run `uv sync` at the repository root, select `.venv/bin/python` as the notebook kernel, and restart the kernel.


In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

mcp_endpoint = f'{apim_resource_gateway_url}/{web_iq_api_path}/mcp'
mcp_headers = entra_mcp_headers or {
    'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
    'x-apikey': web_iq_api_key,
}
mcp_authentication = 'Entra ID' if entra_mcp_headers else 'API key'

async with streamablehttp_client(mcp_endpoint, headers=mcp_headers, timeout=60.0) as streams:
    async with ClientSession(streams[0], streams[1]) as session:
        await session.initialize()
        mcp_tools = (await session.list_tools()).tools

available_mcp_tool_names = {tool.name for tool in mcp_tools}

utils.print_ok(f'Discovered {len(mcp_tools)} Web IQ MCP tool(s) using {mcp_authentication}')
for tool in mcp_tools:
    print(f'  - {tool.name}: {tool.description or "No description"}')

if 'web' not in available_mcp_tool_names:
    raise RuntimeError("The Web IQ account does not expose the required 'web' MCP tool.")


### 7️⃣ Invoke the `web` MCP tool

The Web IQ MCP tools accept the same parameters as their REST counterparts. This call is recorded with `Operation ID = mcp-post` and `MCP Tool = web`. The helper truncates only the displayed output so the notebook remains readable.


In [ ]:
async def call_web_iq_tool(tool_name: str, arguments: dict, headers: dict[str, str]):
    async with streamablehttp_client(mcp_endpoint, headers=headers, timeout=60.0) as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()
            return await session.call_tool(tool_name, arguments)

def render_mcp_content(result, limit: int = 4000) -> str:
    chunks = []
    for item in result.content:
        text = getattr(item, 'text', None)
        chunks.append(text if text is not None else str(item))
    rendered = '\n'.join(chunks)
    return rendered if len(rendered) <= limit else rendered[:limit] + '\n… [display truncated]'

def extract_mcp_web_response(result) -> dict | None:
    structured = getattr(result, 'structuredContent', None)
    if isinstance(structured, dict) and isinstance(structured.get('webResults'), list):
        return structured
    for item in result.content:
        text = getattr(item, 'text', None)
        if not text:
            continue
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict) and isinstance(payload.get('webResults'), list):
            return payload
    return None

mcp_web_result = await call_web_iq_tool(
    'web',
    {
        'query': 'What is Azure API Management and how does it support AI gateways?',
        'maxResults': 3,
        'maxLength': 3000,
        'contentFormat': 'markdown',
    },
    mcp_headers,
)

if getattr(mcp_web_result, 'isError', False):
    raise RuntimeError(render_mcp_content(mcp_web_result))

mcp_web_response = extract_mcp_web_response(mcp_web_result)
if mcp_web_response:
    print(json.dumps(summarize_web_response(mcp_web_response), indent=2))
else:
    print(render_mcp_content(mcp_web_result))


### 8️⃣ Verify the MCP Browse denial

Web IQ may advertise `browse` in `tools/list`, but APIM inspects `tools/call` and rejects that tool with `403 Forbidden` before it reaches Web IQ. The policy records the attempt with `MCP Tool = browse` without logging the requested URL.


In [ ]:
def exception_messages(error: BaseException) -> list[str]:
    nested = getattr(error, 'exceptions', None)
    if nested:
        messages = []
        for child in nested:
            messages.extend(exception_messages(child))
        return messages
    return [f'{type(error).__name__}: {error}']

mcp_browse_was_blocked = False
try:
    await call_web_iq_tool(
        'browse',
        {
            'url': 'https://news.microsoft.com/source/',
            'contentFormat': 'markdown',
            'maxLength': 3000,
        },
        mcp_headers,
    )
except Exception as error:
    mcp_browse_was_blocked = True
    utils.print_ok('The Browse MCP tool was blocked by APIM as expected')
    print('\n'.join(exception_messages(error))[:1200])

assert mcp_browse_was_blocked, 'Browse unexpectedly passed through APIM. Verify that the latest policy is deployed.'


<a id='metrics'></a>
### 9️⃣ Query usage metrics in Application Insights

Custom metrics can take several minutes to arrive. If the tables are empty, wait briefly and rerun these cells. The first query reports request volume and blocked calls by APIM subscription, operation, authentication mode, and MCP tool.


In [ ]:
import pandas as pd
import shlex

def query_application_insights(kql: str) -> pd.DataFrame:
    result = utils.run(
        f'az monitor app-insights query --app {application_insights_name} '
        f'--resource-group {resource_group_name} --analytics-query {shlex.quote(kql)}',
        'Application Insights query succeeded',
        'Application Insights query failed',
    )
    if not result.success or not result.json_data.get('tables'):
        return pd.DataFrame()
    table = result.json_data['tables'][0]
    return pd.DataFrame(
        table.get('rows', []),
        columns=[column['name'] for column in table.get('columns', [])],
    )

usage_query = r'''
customMetrics
| where timestamp > ago(1h) and name in ('Web IQ Requests', 'Web IQ Blocked Requests')
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend McpTool = tostring(dimensions['MCP Tool'])
| summarize Requests = sumif(value, name == 'Web IQ Requests'), BlockedRequests = sumif(value, name == 'Web IQ Blocked Requests')
    by SubscriptionId, OperationId, Authentication, McpTool
| order by Requests desc
'''

usage_df = query_application_insights(usage_query)
usage_df


The latency query groups completed calls by consumer, operation, MCP tool, and upstream HTTP status. This makes throttling and service errors visible without capturing request or response bodies.


In [ ]:
latency_query = r'''
customMetrics
| where timestamp > ago(1h) and name == 'Web IQ Latency'
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend McpTool = tostring(dimensions['MCP Tool'])
| extend StatusCode = tostring(dimensions['Status Code'])
| summarize Calls = count(), AverageLatencyMs = round(avg(value), 1), P95LatencyMs = round(percentile(value, 95), 1)
    by SubscriptionId, OperationId, Authentication, McpTool, StatusCode
| order by Calls desc
'''

latency_df = query_application_insights(latency_query)
latency_df


### Exercise — verify that Browse is blocked

Use the second APIM subscription to call `/browse`. Before running the next cell, predict the status and response body. The request should return APIM's structured `403` without reaching Web IQ. After telemetry arrives, rerun the usage query above to see `browse-url` with one blocked request.


In [ ]:
# Answer scaffold: change target_url to verify that every Browse target is blocked.
def browse_with_subscription(target_url: str, subscription_index: int = 1) -> requests.Response:
    subscription = apim_subscriptions[subscription_index]
    response = requests.post(
        f'{apim_resource_gateway_url}/{web_iq_api_path}/browse',
        headers={
            'Ocp-Apim-Subscription-Key': subscription['key'],
            'x-apikey': web_iq_api_key,
            'content-type': 'application/json',
        },
        json={
            'url': target_url,
            'contentFormat': 'markdown',
            'maxLength': 3000,
            'liveCrawl': 'fallback',
        },
        timeout=30,
    )
    utils.print_response_code(response)
    print(json.dumps(response.json(), indent=2))
    return response

browse_response = browse_with_subscription('https://news.microsoft.com/source/')
assert browse_response.status_code == 403
assert browse_response.json()['errorCode'] == 'BrowseOperationBlocked'


### Pitfalls and extensions

- **Telemetry delay:** Application Insights custom metrics are not immediate; rerun the queries after a few minutes.
- **Credential boundaries:** Clients send an APIM subscription key plus either a Web IQ `x-apikey` or bearer token. APIM strips its subscription credential before forwarding and never stores the Web IQ API key.
- **Metric cardinality:** Keep dimensions bounded. Search text, URLs, trace IDs, and client IP addresses are intentionally excluded.
- **Policy scope:** The block checks APIM's stable REST operation ID (`browse-url`) and the MCP `tools/call` name (`browse`). Remove or revise the `choose` block in [policy.xml](policy.xml) only when Browse is approved for your consumers.
- **MCP reuse:** The `list_web_iq_tools` and `call_web_iq_tool` helpers can be reused by agent code that supplies the same APIM and Web IQ credentials.
- **Two kinds of instrumentation:** These APIM metrics measure gateway usage. Web IQ [instrumentation](https://webiq.microsoft.ai/documentation/instrumentation/) separately records which citations an LLM uses and which links users click.
- **Extension:** Add APIM rate limits per subscription, or proxy other Web IQ verticals after confirming that your Web IQ account permits them.


<a id='portal'></a>
### View metrics in the Azure portal

Open the deployed Application Insights resource, select **Metrics**, choose the `web-iq` custom namespace, and select **Web IQ Requests**, **Web IQ Blocked Requests**, or **Web IQ Latency**. Split by **Subscription ID**, **Operation ID**, **Authentication**, **MCP Tool**, or **Status Code**.


<a id='clean-up'></a>
### 🗑️ Clean up resources

Run [clean-up-resources.ipynb](clean-up-resources.ipynb) when finished to remove the resource group and avoid further charges.
